# Paso 4: Operacionalización y Despliegue del Modelo

En este notebook se carga el modelo entrenado en el **Notebook 3: Construcción del Modelo** y el conjunto de datos generado en el **Notebook 2: Ingeniería de Características** para crear los artefactos necesarios para el despliegue. Se implementan y validan localmente las funciones de inferencia, y posteriormente se empaquetan junto con el modelo y el esquema de entrada para su despliegue en **Azure Machine Learning**.

Finalmente, el paquete se almacena en **Azure Blob Storage** y se utiliza para desplegar un **endpoint en tiempo real (Real-time Endpoint)**, permitiendo realizar predicciones en un entorno de producción.

> **Nota:** La ejecución completa de este notebook toma aproximadamente **1 minuto**, dependiendo de la configuración del recurso de cómputo utilizado.

In [1]:
from pathlib import Path
import joblib
import pandas as pd

In [2]:
# Carpeta raíz del proyecto
BASE_PATH = Path.cwd().parent

# Carpetas del proyecto
INPUT_PATH = BASE_PATH / "data" / "input_data"
OUTPUT_PATH = BASE_PATH / "data" / "output_data"
MODEL_PATH = BASE_PATH / "models"

In [3]:
print(BASE_PATH)
print(INPUT_PATH)
print(OUTPUT_PATH)
print(MODEL_PATH)

g:\Mi unidad\Trabajo\Cursos\sabana\IA_Siemens\clase2\PredictiveMaintenance
g:\Mi unidad\Trabajo\Cursos\sabana\IA_Siemens\clase2\PredictiveMaintenance\data\input_data
g:\Mi unidad\Trabajo\Cursos\sabana\IA_Siemens\clase2\PredictiveMaintenance\data\output_data
g:\Mi unidad\Trabajo\Cursos\sabana\IA_Siemens\clase2\PredictiveMaintenance\models


In [4]:
MODEL_FILE = MODEL_PATH / "random_forest_model.pkl"

model = joblib.load(MODEL_FILE)

print("Modelo cargado correctamente.")

Modelo cargado correctamente.


In [5]:
print(model)

RandomForestClassifier(max_depth=15, n_estimators=50, n_jobs=-1,
                       random_state=42)


## Cargar los datos para inferencia

Antes de desplegar un modelo es recomendable validar que funciona correctamente con datos reales.

En este paso se carga el conjunto de datos generado durante la etapa de ingeniería de características y se seleccionan algunas observaciones para realizar predicciones de forma local.

In [6]:
# Cargar el conjunto de datos preparado
data = pd.read_parquet(
    OUTPUT_PATH / "featureengineering_files.parquet"
)

print(f"Registros: {data.shape[0]}")
print(f"Variables: {data.shape[1]}")

Registros: 73143
Variables: 43


## Preparar los datos de entrada

Para realizar inferencias únicamente se utilizan las variables predictoras.

La columna **label_e** corresponde a la variable objetivo utilizada durante el entrenamiento y, por lo tanto, no debe enviarse al modelo durante la predicción.

In [7]:
X = data.drop(columns=["label_e"])

X.head()

,machineID,dt_truncated,datetime,volt_rollingmean_12,volt_rollingstd_12,rotate_rollingmean_12,rotate_rollingstd_12,pressure_rollingmean_12,pressure_rollingstd_12,vibration_rollingmean_12,...,comp1sum,comp2sum,comp3sum,comp4sum,age,model_model1,model_model2,model_model3,model_model4,failure
0,1,2015-01-01 00:00:00,2015-01-01 08:30:00,169.512597,6.239068,425.853115,46.581966,101.886267,13.166950,41.119335,...,18.0,213.0,153.0,168.0,18,0,0,1,0,0.0
1,1,2015-01-01 12:00:00,2015-01-01 17:30:00,167.007032,8.260186,434.948972,50.000568,100.970047,10.863588,39.412711,...,19.0,214.0,154.0,169.0,18,0,0,1,0,0.0
2,1,2015-01-02 00:00:00,2015-01-02 05:30:00,171.754734,13.479925,454.534311,44.925868,93.989345,7.963589,40.265190,...,19.0,214.0,154.0,169.0,18,0,0,1,0,0.0
3,1,2015-01-02 12:00:00,2015-01-02 17:30:00,168.184995,15.356860,444.596343,39.260749,100.778017,10.107193,39.209928,...,20.0,215.0,155.0,170.0,18,0,0,1,0,0.0
4,1,2015-01-03 00:00:00,2015-01-03 05:30:00,172.717240,12.879603,468.079805,40.727721,99.586915,12.005493,40.406556,...,20.0,215.0,155.0,170.0,18,0,0,1,0,0.0


## Realizar una predicción local

Antes de desplegar el modelo en Azure Machine Learning, se realiza una prueba local utilizando algunas observaciones del conjunto de datos.

Esta validación permite verificar que el modelo fue cargado correctamente y que genera predicciones de forma adecuada.

In [10]:
# Variables utilizadas durante el entrenamiento
remove_names = [
    "datetime",
    "machineID",
    "dt_truncated",
    "failure",
#    "model",
    "label_e"
]

# Preparar los datos para inferencia
X = data.drop(columns=remove_names)

X.head()

,volt_rollingmean_12,volt_rollingstd_12,rotate_rollingmean_12,rotate_rollingstd_12,pressure_rollingmean_12,pressure_rollingstd_12,vibration_rollingmean_12,vibration_rollingstd_12,volt_rollingmean_24,volt_rollingstd_24,...,error5sum_rollingmean_24,comp1sum,comp2sum,comp3sum,comp4sum,age,model_model1,model_model2,model_model3,model_model4
0,169.512597,6.239068,425.853115,46.581966,101.886267,13.166950,41.119335,4.473233,169.512597,6.239068,...,0.0,18.0,213.0,153.0,168.0,18,0,0,1,0
1,167.007032,8.260186,434.948972,50.000568,100.970047,10.863588,39.412711,5.716674,167.101438,7.975963,...,0.0,19.0,214.0,154.0,169.0,18,0,0,1,0
2,171.754734,13.479925,454.534311,44.925868,93.989345,7.963589,40.265190,5.761581,169.467283,11.387289,...,0.0,19.0,214.0,154.0,169.0,18,0,0,1,0
3,168.184995,15.356860,444.596343,39.260749,100.778017,10.107193,39.209928,5.957205,169.969864,14.655751,...,0.0,20.0,215.0,155.0,170.0,18,0,0,1,0
4,172.717240,12.879603,468.079805,40.727721,99.586915,12.005493,40.406556,5.574350,170.451117,14.671892,...,0.0,20.0,215.0,155.0,170.0,18,0,0,1,0


> **Importante:** Durante la etapa de inferencia, el modelo debe recibir **exactamente las mismas variables predictoras** con las que fue entrenado. Si se incluyen variables adicionales, se omiten variables o cambia el orden de las columnas, **Scikit-Learn** generará un error para evitar realizar predicciones sobre datos inconsistentes.

In [11]:
sample = X.iloc[:5]

predictions = model.predict(sample)

predictions

array([0., 0., 0., 0., 0.])

In [12]:
results = sample.copy()

results["Predicción"] = predictions.astype(int)

results

,volt_rollingmean_12,volt_rollingstd_12,rotate_rollingmean_12,rotate_rollingstd_12,pressure_rollingmean_12,pressure_rollingstd_12,vibration_rollingmean_12,vibration_rollingstd_12,volt_rollingmean_24,volt_rollingstd_24,...,comp1sum,comp2sum,comp3sum,comp4sum,age,model_model1,model_model2,model_model3,model_model4,Predicción
0,169.512597,6.239068,425.853115,46.581966,101.886267,13.166950,41.119335,4.473233,169.512597,6.239068,...,18.0,213.0,153.0,168.0,18,0,0,1,0,0
1,167.007032,8.260186,434.948972,50.000568,100.970047,10.863588,39.412711,5.716674,167.101438,7.975963,...,19.0,214.0,154.0,169.0,18,0,0,1,0,0
2,171.754734,13.479925,454.534311,44.925868,93.989345,7.963589,40.265190,5.761581,169.467283,11.387289,...,19.0,214.0,154.0,169.0,18,0,0,1,0,0
3,168.184995,15.356860,444.596343,39.260749,100.778017,10.107193,39.209928,5.957205,169.969864,14.655751,...,20.0,215.0,155.0,170.0,18,0,0,1,0,0
4,172.717240,12.879603,468.079805,40.727721,99.586915,12.005493,40.406556,5.574350,170.451117,14.671892,...,20.0,215.0,155.0,170.0,18,0,0,1,0,0
